RAG SYSTEM

In [5]:
!pip install sentence-transformers chromadb groq pandas -q
print('Installation Completed!!!')

Installation Completed!!!


In [6]:
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from groq import Groq
import os
print(f'All Libraries Successfully!!!!')

All Libraries Successfully!!!!


In [7]:
GROQ_API_KEY="XXXX"
os.environ["GROQ_API_KEY"]=GROQ_API_KEY #os.environ acts like locker key,when someone see our project it doesn't show our api key
groq_client=Groq(api_key=GROQ_API_KEY)
print('Groq API client initialised')
print('Note:If you see an authentication error later,double check your API Key')

Groq API client initialised
Note:If you see an authentication error later,double check your API Key


In [8]:
df=pd.read_csv('college_notes.csv')
print("Shape of Data Set:",df.shape)
print("\nColumn Names:",df.columns.tolist())
print("\nFirst 3 Rows:")
print(df.head(3))

Shape of Data Set: (15, 4)

Column Names: ['note_id', 'subject', 'topic', 'content']

First 3 Rows:
  note_id           subject          topic  \
0    N001  Data Engineering  ETL Pipelines   
1    N002  Data Engineering  SQL Databases   
2    N003  Data Engineering  Data Cleaning   

                                             content  
0  ETL stands for Extract Transform Load. It is t...  
1  A database is an organized collection of data ...  
2  Data cleaning involves fixing or removing inco...  


In [9]:
df=pd.read_csv('college_notes.csv')
print("Shape of Data Set:",df.shape)
print("\nColumn Names:",df.columns.tolist())
print("\nFirst 3 Rows:")
print(df.head(3).to_string(index=False))

Shape of Data Set: (15, 4)

Column Names: ['note_id', 'subject', 'topic', 'content']

First 3 Rows:
note_id          subject         topic                                                                                                                                                                                                                  content
   N001 Data Engineering ETL Pipelines ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it into a clean and structured format and loading it into a database or data warehouse for analysis.
   N002 Data Engineering SQL Databases        A database is an organized collection of data stored electronically. SQL or Structured Query Language is used to interact with relational databases. Common SQL commands include SELECT INSERT UPDATE and DELETE.
   N003 Data Engineering Data Cleaning       Data cleaning involves fixing or removing incorrect incomplete duplicate or corrupted d

In [10]:
print("Subjects in the datset:")
print(df['subject'].value_counts())
print("\nSample of topics:")
print(df[['note_id','subject','topic']].to_string(index=False))
print("\nLength of content (numbers of characters) for each note:")
df['content_length']=df['content'].apply(len)
print(df[['topic','content_length']].to_string(index=False))

Subjects in the datset:
subject
Data Engineering      5
Machine Learning      5
Generative AI         3
Python Programming    2
Name: count, dtype: int64

Sample of topics:
note_id            subject                          topic
   N001   Data Engineering                  ETL Pipelines
   N002   Data Engineering                  SQL Databases
   N003   Data Engineering                  Data Cleaning
   N004   Data Engineering       APIs and Data Collection
   N005   Data Engineering           Big Data and PySpark
   N006   Machine Learning            Supervised Learning
   N007   Machine Learning               Model Evaluation
   N008   Machine Learning            Feature Engineering
   N009   Machine Learning                 Decision Trees
   N010   Machine Learning                  Random Forest
   N011      Generative AI          Large Language Models
   N012      Generative AI             Prompt Engineering
   N013      Generative AI Retrieval Augmented Generation
   N014 Python 

In [11]:
#convert to chunks
documents=df['content'].tolist()
ids=[f"note_{row['note_id']}" for row in df.to_dict('records')]
metadatas=[
    {"subject":row['subject'],"topic":row['topic']}
    for row in df.to_dict('records')
]
print(f"Total Chunks Prepared:{len(documents)}")
print(f"First document's ID:{ids[3]}")
print(f"First document's Metadata:{metadatas[3]}")
print(f"First 100 characters of document:{documents[3][:100]}....")


Total Chunks Prepared:15
First document's ID:note_N004
First document's Metadata:{'subject': 'Data Engineering', 'topic': 'APIs and Data Collection'}
First 100 characters of document:An API or Application Programming Interface allows two software applications to talk to each other. ....


In [12]:
print("Loading Embedding Model..........")
embedding_model=SentenceTransformer('all-MiniLM-L6-v2')
print('Embedding Model Loaded successfully!!!!')
test_embedding=embedding_model.encode("This is the Test Sentence")
print(f'Test Embedding Shape:{test_embedding.shape}')
print(f"First 5 values of test embedding:{test_embedding[:5]}")

Loading Embedding Model..........


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding Model Loaded successfully!!!!
Test Embedding Shape:(384,)
First 5 values of test embedding:[ 0.07163641  0.07820371 -0.0090355   0.08910967  0.03642739]


In [13]:
print("Loading Embedding Model..........")
embedding_model=SentenceTransformer('all-MiniLM-L6-v2')
print('Embedding Model Loaded successfully!!!!')
test_embedding=embedding_model.encode("Hirdunaria granulosa is the botanical name of Earth Worm")
print(f'Test Embedding Shape:{test_embedding.shape}')
print(f"First 5 values of test embedding:{test_embedding[:5]}")

Loading Embedding Model..........


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding Model Loaded successfully!!!!
Test Embedding Shape:(384,)
First 5 values of test embedding:[ 0.02668911  0.05166563 -0.0030325  -0.02047203 -0.03223315]


In [14]:
chroma_client=chromadb.Client()
collection=chroma_client.get_or_create_collection(name="new_college_notes_rag")
print("ChromaDB client created.")
print(f"Collection name:new_college_notes_rag")
print(f"Documents in collection so far:{collection.count()}")

ChromaDB client created.
Collection name:new_college_notes_rag
Documents in collection so far:0


In [15]:
print("Generating Embeddings for all 15 notes")
embeddings=embedding_model.encode(documents,show_progress_bar=True) #show_progress_bar=True-->used for the green bar in output(optional)
print(f"\nEmbedding matrix Shape:{embeddings.shape}")
embeddings_list=embeddings.tolist()
collection.add(
    documents=documents,
    embeddings=embeddings_list,
    ids=ids,
    metadatas=metadatas
)
print(f"\nDocuments Added Successfully in ChromaDB!!!!")
print(f"Total documents in collection:{collection.count()}")

Generating Embeddings for all 15 notes


Batches:   0%|          | 0/1 [00:00<?, ?it/s]


Embedding matrix Shape:(15, 384)

Documents Added Successfully in ChromaDB!!!!
Total documents in collection:15


In [16]:
#creating the chunks and retrieve
def retrieve_relevant_chunks(question,top_k=3):
  """
  Given a user question,retrieve the most relevant document chunks from ChromaDB
  Parameters:
  question(str):The user's question as a text string
  top_k(int):How many top results to return(default:3)
  Returns:
  A dictionary containing retrieved documents,distances and metadata
  """
  question_embedding=embedding_model.encode(question).tolist()
  results=collection.query(
      query_embeddings=[question_embedding],
      n_results=top_k,
  )
  return results
  print("Retrieval Function defined successfully")
  print("Function:retrieve_relevant_chunks(question,top_k=3)")

In [17]:
test_question="What is ETL and how does it work in data engineering"
print(f"Test Question:{test_question}")
results=retrieve_relevant_chunks(test_question,top_k=3)
print("\nTop 3 Retrieved Chunks:")
for i,(doc,dist,meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
    )):
  print(f"\nResults{i+1}:")
  print(f"-->Subject:{meta['subject']}")
  print(f"-->Topic:{meta['topic']}")
  print(f"-->Distance:{dist:.4f}")
  print(f"-->Content:{doc[:120]}.....")

Test Question:What is ETL and how does it work in data engineering

Top 3 Retrieved Chunks:

Results1:
-->Subject:Data Engineering
-->Topic:ETL Pipelines
-->Distance:0.2041
-->Content:ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it i.....

Results2:
-->Subject:Data Engineering
-->Topic:APIs and Data Collection
-->Distance:1.1100
-->Content:An API or Application Programming Interface allows two software applications to talk to each other. In data engineering .....

Results3:
-->Subject:Python Programming
-->Topic:Data Visualization
-->Distance:1.3892
-->Content:Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplo.....


In [18]:
test_question="What is ETL and how does it work in data engineering"
print(f"Test Question:{test_question}")
results=retrieve_relevant_chunks(test_question,top_k=5)
print("\nTop 3 Retrieved Chunks:")
for i,(doc,dist,meta) in enumerate(zip(
    results['documents'][0],
    results['distances'][0],
    results['metadatas'][0]
    )):
  print(f"\nResults{i+1}:")
  print(f"-->Subject:{meta['subject']}")
  print(f"-->Topic:{meta['topic']}")
  print(f"-->Distance:{dist:.4f}")
  print(f"-->Content:{doc[:120]}.....")

Test Question:What is ETL and how does it work in data engineering

Top 3 Retrieved Chunks:

Results1:
-->Subject:Data Engineering
-->Topic:ETL Pipelines
-->Distance:0.2041
-->Content:ETL stands for Extract Transform Load. It is the process of collecting raw data from different sources transforming it i.....

Results2:
-->Subject:Data Engineering
-->Topic:APIs and Data Collection
-->Distance:1.1100
-->Content:An API or Application Programming Interface allows two software applications to talk to each other. In data engineering .....

Results3:
-->Subject:Python Programming
-->Topic:Data Visualization
-->Distance:1.3892
-->Content:Data visualization is the process of representing data as charts graphs and visual formats. Python libraries like Matplo.....

Results4:
-->Subject:Data Engineering
-->Topic:SQL Databases
-->Distance:1.4165
-->Content:A database is an organized collection of data stored electronically. SQL or Structured Query Language is used to interac.....

Results5:
-->Subj

In [26]:
def build_context_from_results(results):
  """
  Format ChromaDB retrieval results into a readable context string.
  Parameters:
  results:The output from collection.query()-a dictionary
  Returns:
  context_str(str):A formatted string od all retrieved document chunks
  """
  context_parts=[] #empty list to collect formatted chunks
  for i,(doc,meta) in enumerate(zip(
      results['documents'][0],
      results['metadatas'][0]
  )):
    chunk_text=f"[Source{i+1}:{meta['subject']}-{meta['topic']}]\n{doc}"
    context_parts.append(chunk_text)
  return "\n\n".join(context_parts)

In [30]:
def generate_rag_answer(question, context):
    """
    Send the retrieved context and question to the Groq LLM for answer generation.
    """
    system_prompt="""You are a helpful academic assistant for engineering students.
You will be given context retrieved from a college knowledge base and a student's question.
RULES:
1. Answer ONLY using the information provided in the context below.
2. If the answer is not found in the context say exactly:
   "I don't have enough information in my knowledge base to answer this question."
3. Do not use your general training knowledge.
4. Keep answers clear, accurate, and beginner-friendly.
5. Mention which source the information comes from when possible.
"""
    user_prompt = f"""Context from Knowledge Base:
{context}
Question:
{question}
"""
    response=groq_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.1,
        max_tokens=500
    )
    answer=response.choices[0].message.content
    return answer
print("RAG generation defined successfully!!")

RAG generation defined successfully!!


In [34]:
def ask_college_assistant(question,top_k=3,verbose=True):
    """
    Complete RAG pipeline: Given a question, retrieve relevant context and generate an answer.
    Parameters:
    question(str):The user's question
    top_k(int):Number of chunks to retrieve (default: 3)
    verbose(bool):Whether to print intermediate steps (default: True)
    Returns:
    answer(str):The final generated answer
    """
    if verbose:
        print(f"Question: {question}")
    # Step 1: Retrieve relevant chunks
    results=collection.query(
        query_texts=[question],
        n_results=top_k
    )
    if verbose:
        print(f"\nRetrieved {len(results['documents'][0])} chunks")
    # Step 2: Build context from retrieved chunks
    context=build_context_from_results(results)
    if verbose:
        print("\nContext Built Successfully")
        print("\nContext:")
        print(context)
    # Step 3: Generate answer using LLM
    answer=generate_rag_answer(question, context)
    if verbose:
        print("\nAnswer Generated Successfully")
    return answer
print("RAG pipeline defined successfully")

RAG pipeline defined successfully
